<a href="https://colab.research.google.com/github/devmurarijay13/pytorch-deep-learning/blob/main/cat_vs_dog_classifier.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
pip install torchinfo

In [2]:
import torch
import torch.nn as nn
from torch.utils.data import Dataset,DataLoader
import torch.optim as optim
from torchvision import datasets, transforms, models
from torchinfo import summary

from sklearn.metrics import classification_report,confusion_matrix

In [3]:
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

In [4]:
!kaggle datasets download salader/dogsvscats

Dataset URL: https://www.kaggle.com/datasets/salader/dogsvscats
License(s): unknown
100% 1.06G/1.06G [00:18<00:00, 63.1MB/s]



In [5]:
import zipfile
zip_ref = zipfile.ZipFile('/content/dogsvscats.zip', 'r')
zip_ref.extractall('/content')
zip_ref.close()

In [6]:
import warnings
warnings.filterwarnings('ignore')

In [7]:
torch.manual_seed(42)

In [8]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'using device : {device}')

using device : cuda


In [10]:
train_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

test_transform = transforms.Compose([
    transforms.Resize((128,128)),
    transforms.ToTensor(),
    transforms.Normalize([0.5]*3, [0.5]*3)
])

In [11]:
train_data = datasets.ImageFolder(
    root='/content/catsvsdogs/train',
    transform=train_transform
)

test_data = datasets.ImageFolder(
    root='/content/catsvsdogs/test',
    transform=test_transform
)

In [12]:
train_loader = DataLoader(train_data,batch_size=64,shuffle=True,pin_memory=True,num_workers=4)
test_loader = DataLoader(test_data,batch_size=64,shuffle=False,pin_memory=True,num_workers=4)

In [13]:
learning_rate = 0.001
epochs = 50

In [22]:
class ConvolutionalNeuralNetwork(nn.Module):

    def __init__(self,in_channels):

        super().__init__()

        self.feature_extractor = nn.Sequential(

            nn.Conv2d(in_channels,32,kernel_size=3,padding=1),
            nn.BatchNorm2d(32),
            nn.LeakyReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2),

            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.BatchNorm2d(64),
            nn.LeakyReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2),

            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(),
            nn.MaxPool2d(kernel_size=2,stride=2)
        )


        self.classifier = nn.Sequential(
            nn.Flatten(),

            nn.Linear(32768,16),
            nn.SELU(),
            nn.BatchNorm1d(16),
            nn.Dropout(0.2),

            nn.Linear(16,8),
            nn.SELU(),
            nn.BatchNorm1d(8),
            nn.Dropout(0.2),

            nn.Linear(8,2)
        )

    def forward(self,x):
        features = self.feature_extractor(x)
        outputs = self.classifier(features)

        return outputs

In [23]:
model = ConvolutionalNeuralNetwork(3)

model = model.to(device)

optimizer = optim.Adam(model.parameters(),lr=learning_rate,weight_decay=1e-3)
criterion = nn.CrossEntropyLoss()

In [24]:
summary(model)

Layer (type:depth-idx)                   Param #
ConvolutionalNeuralNetwork               --
├─Sequential: 1-1                        --
│    └─Conv2d: 2-1                       896
│    └─BatchNorm2d: 2-2                  64
│    └─LeakyReLU: 2-3                    --
│    └─MaxPool2d: 2-4                    --
│    └─Conv2d: 2-5                       18,496
│    └─BatchNorm2d: 2-6                  128
│    └─LeakyReLU: 2-7                    --
│    └─MaxPool2d: 2-8                    --
│    └─Conv2d: 2-9                       73,856
│    └─BatchNorm2d: 2-10                 256
│    └─LeakyReLU: 2-11                   --
│    └─MaxPool2d: 2-12                   --
├─Sequential: 1-2                        --
│    └─Flatten: 2-13                     --
│    └─Linear: 2-14                      524,304
│    └─SELU: 2-15                        --
│    └─BatchNorm1d: 2-16                 32
│    └─Dropout: 2-17                     --
│    └─Linear: 2-18                      136
│    └─SEL

In [25]:

model.train()

train_losses = []
train_accuracies = []

for epoch in range(epochs):
    running_loss = 0.0
    correct = 0
    total = 0

    for batch_images,batch_labels in train_loader:

        batch_images,batch_labels = batch_images.to(device),batch_labels.to(device)

        optimizer.zero_grad()

        outputs = model(batch_images)

        loss = criterion(outputs,batch_labels)

        loss.backward()

        optimizer.step()

        # Statistics
        running_loss += loss.item()
        _, predicted = torch.max(outputs.data, 1)
        total += batch_labels.size(0)
        correct += (predicted == batch_labels).sum().item()

        # Print progress
        if (epoch + 1) % 200 == 0:
            print(f'Epoch [{epoch+1}/{epochs}], Step [{epoch+1}/{len(train_loader)}], '
            f'Loss: {loss.item():.4f}, Accuracy: {100 * correct / total:.2f}%')

    # Calculate epoch statistics
    epoch_loss = running_loss / len(train_loader)
    epoch_accuracy = 100 * correct / total
    train_losses.append(epoch_loss)
    train_accuracies.append(epoch_accuracy)

    print(f'Epoch [{epoch+1}/{epochs}] completed: '
            f'Average Loss: {epoch_loss:.4f}, Accuracy: {epoch_accuracy:.2f}%')

Epoch [1/50] completed: Average Loss: 0.5843, Accuracy: 68.98%
Epoch [2/50] completed: Average Loss: 0.4671, Accuracy: 78.55%
Epoch [3/50] completed: Average Loss: 0.4223, Accuracy: 81.27%
Epoch [4/50] completed: Average Loss: 0.4011, Accuracy: 82.62%
Epoch [5/50] completed: Average Loss: 0.3799, Accuracy: 83.69%
Epoch [6/50] completed: Average Loss: 0.3630, Accuracy: 84.25%
Epoch [7/50] completed: Average Loss: 0.3466, Accuracy: 85.44%
Epoch [8/50] completed: Average Loss: 0.3310, Accuracy: 86.34%
Epoch [9/50] completed: Average Loss: 0.3188, Accuracy: 86.70%
Epoch [10/50] completed: Average Loss: 0.3041, Accuracy: 87.56%
Epoch [11/50] completed: Average Loss: 0.2882, Accuracy: 88.27%
Epoch [12/50] completed: Average Loss: 0.2725, Accuracy: 89.16%
Epoch [13/50] completed: Average Loss: 0.2702, Accuracy: 88.89%
Epoch [14/50] completed: Average Loss: 0.2619, Accuracy: 89.47%
Epoch [15/50] completed: Average Loss: 0.2518, Accuracy: 90.03%
Epoch [16/50] completed: Average Loss: 0.2402, Ac

In [26]:
model.eval()

total = 0
correct = 0

with torch.no_grad():
    for batch_images,batch_labels in test_loader:
        batch_images,batch_labels = batch_images.to(device),batch_labels.to(device)

        outputs = model(batch_images)

        _,predicted = torch.max(outputs,1)

        total += batch_labels.shape[0]

        correct += (predicted == batch_labels).sum().item()

print(f"Test Accuracy : {correct/total}")

Test Accuracy : 0.9244


In [27]:
# model.eval()

total = 0
correct = 0

with torch.no_grad():
    for batch_images,batch_labels in train_loader:
        batch_images,batch_labels = batch_images.to(device),batch_labels.to(device)

        outputs = model(batch_images)

        _,predicted = torch.max(outputs,1)

        total += batch_labels.shape[0]

        correct += (predicted == batch_labels).sum().item()

print(f"Train Accuracy : {correct/total}")

Train Accuracy : 0.94745
